# Lab 01 · Prompting and small evaluations

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## Prompts, memory and evaluation
**Start with a problem:** If every question repeats the assistant’s instructions, changing those instructions means editing each prompt. A follow-up may also lack the previous answer.

**In this lab:** Separate reusable system instructions from user questions, carry a previous response forward, and check the result.

**Goal:** Learn to send system and user prompts, reuse a previous response, validate an answer, and compare prompting and reasoning modes.

ShopEasy has web and mobile clients. An **API gateway** can provide shared routing and security at the edge; a **backend for frontend (BFF)** tailors an API to a particular client. A system can use both. We will use these ideas in short architecture questions, then try two small reasoning puzzles.

The model can make mistakes. Practice mode uses scripted replies, so it teaches the code flow; use live mode to observe real model answers.

**At a glance**

![Lab 01 at a glance: Architecture question (gateway or BFF), then Prompt (system + user), then Remember (pass prior answer), then Validate (JSON + decision), then Compare (older + thinking modes).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/01.svg)

## Think about the example

An API gateway can route requests and apply common policies. A backend for frontend (BFF) can tailor an API to one client, such as a mobile app. A system may use both.

**Concepts:** API gateway; Backend for frontend (BFF); Client-specific APIs.

Use the scenario above to consider these questions before running the code:

1. What shared needs can an API gateway address?
2. Why might mobile and web clients benefit from different BFFs?

## The idea in context

A system message guides the assistant; a user message requests a task. A follow-up can include a previous answer as temporary context.

Parse and validate replies before using them. An older model and DeepSeek thinking modes can be compared on small, checkable examples.

| Key term | Meaning |
| --- | --- |
| **System prompt** | Instructions that guide the assistant’s task and response. |
| **User prompt** | The specific question or request sent by the user. |
| **Temporary context** | Information, such as a previous answer, sent again in a later request. |
| **Validation** | Checking that a response has the expected format and meets a specific condition. |
| **Thinking mode** | A model setting that enables additional internal reasoning before the final answer. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice mode uses scripted answers without a key. For live mode, save `DEEPSEEK_API_KEY` in Colab Secrets, keep `PROVIDER="deepseek"`, set `MODE="live"`, and rerun this setup cell. Live Step 4 downloads a small local model; Step 5 uses DeepSeek thinking mode. Other providers work in Steps 1–3; Step 5 requires DeepSeek. Each client stops after 12 calls.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
MODE = "practice"  # change to "live" for real model calls
PROVIDER = "deepseek"  # or "gemini" or "freeinference" for Steps 1–3
if MODE == "live" and find_spec("transformers") is None:
    %pip install -q "transformers>=4.45,<5" torch
from coursekit.runtime import Client, secret
from coursekit.notebook import PROVIDERS
settings = PROVIDERS[PROVIDER]
key = secret(settings["secret_name"]) if MODE == "live" else ""
plain = Client(mode=MODE, model=settings["model"], base_url=settings["base_url"],
               api_key=key, max_calls=12, extra_body=settings.get("extra_body"))
thinking = None
if PROVIDER == "deepseek":
    thinking = Client(mode=MODE, model=settings["model"], base_url=settings["base_url"],
                      api_key=key, max_calls=12, max_tokens=4096, timeout=120,
                      extra_body={"thinking": {"type": "enabled"}, "reasoning_effort": "low"})

## Step 1 · From one mixed prompt to two roles
Start with **one mixed user prompt**: the assistant's role, response style, and the architecture question are all in the same text. It can work, but it is harder to maintain when the question changes because the shared instructions and the changing request are mixed together.

Next, split the *same* content into two messages:
- **System prompt:** stable instructions for the assistant, such as its role and response style.
- **User prompt:** the current task and its specific facts or question.

The split makes shared instructions easier to reuse and the question easier to change. On services that support prefix caching, keeping a long shared prefix unchanged **may** reduce repeated input processing and cost. [DeepSeek enables context caching automatically](https://api-docs.deepseek.com/guides/kv_cache/), but hits are best-effort and depend on an eligible matching prefix. This tiny example does **not** measure or guarantee a cache hit; stable text in a user message can also be cached. System instructions do not guarantee correct or safe answers.

In [ ]:
mixed_prompt = """You are a software architecture tutor. Answer in one sentence.
ShopEasy needs a tailored API for its mobile app. What pattern fits?"""
mixed_answer = plain.ask(mixed_prompt, system="", practice="A mobile BFF fits.")

system_prompt = "You are a software architecture tutor. Answer in one sentence."
user_prompt = "ShopEasy needs a tailored API for its mobile app. What pattern fits?"
split_answer = plain.ask(user_prompt, system=system_prompt, practice="A mobile BFF fits.")

new_question = "For several services, which provides one routing entry point: an API gateway or a BFF?"
next_answer = plain.ask(new_question, system=system_prompt, practice="An API gateway fits.")
print("Mixed prompt:", mixed_answer)
print("Split prompt:", split_answer)
print("Same system prompt, new question:", next_answer)

## Step 2 · Temporary memory: pass the previous answer
First ask a follow-up without context. Then save one response in `first_answer` and include it in the next user prompt. The second call can use that text as context. This is temporary memory held in a Python variable: each API request is independent, and the model does not remember the previous request unless we send the context again. It is not persistent memory.

Because this version copies the response into a new **user message**, it is not a complete role-labelled chat history. A production conversation may instead resend selected user and assistant messages as separate roles.

In [ ]:
without_context = plain.ask("What is one drawback of this choice? Answer in one sentence.",
    practice="Please tell me which choice you mean.")

first_answer = plain.ask(
    "ShopEasy's mobile app needs a tailored API. Suggest one pattern in one sentence.",
    practice="Use a mobile BFF to tailor responses to the app.")

follow_up = f"""Earlier answer: {first_answer}
What is one drawback of this choice? Answer in one sentence."""
second_answer = plain.ask(follow_up,
    practice="A BFF adds another component for the team to maintain.")
print("Without context:", without_context)
print("First answer:", first_answer)
print("Follow-up:", second_answer)

## Step 3 · Validate a structured answer
Ask for one JSON object with a decision and a reason. We then check both the **format** and a small expected answer. The scenario explicitly says each client needs its own API, so our classroom answer is `BFF`.

`json.loads` converts JSON text to a Python value. A valid format or matching label does not prove that the reason is correct; read it yourself. The `try` block keeps malformed JSON from crashing the notebook.

In [ ]:
import json

prompt = """Web and mobile clients need different tailored APIs.
Choose BFF or API gateway. Return JSON only, with keys
"decision" and "reason". Give a short reason."""
reply = plain.ask(prompt, practice='{"decision":"BFF","reason":"Each client has different needs."}')
print("Model reply:", reply)
try:
    answer = json.loads(reply)
    valid = (isinstance(answer, dict) and answer.get("decision") in ["BFF", "API gateway"]
             and isinstance(answer.get("reason"), str) and bool(answer["reason"].strip()))
    correct = valid and answer["decision"] == "BFF"
    print("Valid format:", valid, "| Expected decision:", correct)
except json.JSONDecodeError:
    print("Valid format: False | The reply is not JSON")

## Step 4 · An older model: direct and guided prompts
First try an older, small **non-thinking** model, [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct), on two questions. Then give it a more explicit, step-by-step prompt (a simple chain-of-thought technique). There are no prompt-building loops: read and change each prompt directly.

In practice mode the replies below are deliberately scripted, including mistakes. In live mode this step downloads and runs the small model locally on the Colab CPU; the first run needs internet and may take several minutes. Results may vary. A more explicit prompt can help, but it may not correct a small model's mistake.

Here, `9.9` and `9.11` are **decimal numbers**, not software version numbers. The correct answers are **3** and **9.9**.

In [ ]:
if MODE == "live":
    from transformers import pipeline
    old = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)
    def ask_old(prompt, example):
        result = old([{"role": "user", "content": prompt}], max_new_tokens=96, do_sample=False)
        return result[0]["generated_text"][-1]["content"]
else:
    def ask_old(prompt, example):
        return example  # scripted practice response

letters = "How many letter r's are in strawberry?"
decimals = "Which decimal number is larger: 9.9 or 9.11?"
print("Older, direct:", ask_old(letters, "2"))
print("Older, guided:", ask_old("List the letters in strawberry, then count the r's.", "3"))
print("Older, direct:", ask_old(decimals, "9.11"))
print("Older, guided:", ask_old("Write 9.9 and 9.11 with the same decimal places, then compare.", "9.9"))

## Step 5 · Compare DeepSeek without and with thinking
Ask the same two direct questions with DeepSeek's **thinking disabled**, then **thinking enabled**. We do not ask the thinking model to reveal a chain of thought; it can use its own internal reasoning. The helper returns only the final answer.

Both calls use `deepseek-flash`: this is a comparison of two modes of the same model, not an old-versus-new model comparison. The older SmolLM2 experiment above changes the model as well, so differences cannot be attributed to reasoning mode alone. Reasoning mode may use more tokens and time; it can also be wrong. Test rather than assume it always wins.

The thinking-mode API settings below are specific to DeepSeek. If you selected Gemini or FreeInference for Steps 1–3, switch `PROVIDER` to `"deepseek"` and rerun setup to run this step. For the optional live experiment, use a DeepSeek API key in Colab Secrets.

In [ ]:
from decimal import Decimal

if thinking is None:
    print("Switch PROVIDER to deepseek in setup to run the thinking comparison.")
else:
    print("DeepSeek, no thinking:", plain.ask(letters, practice="2"))
    print("DeepSeek, thinking:", thinking.ask(letters, practice="3"))
    print("DeepSeek, no thinking:", plain.ask(decimals, practice="9.11"))
    print("DeepSeek, thinking:", thinking.ask(decimals, practice="9.9"))
print("Python checks: r count =", "strawberry".count("r"))
print("Python checks: 9.9 > 9.11 is", Decimal("9.9") > Decimal("9.11"))

## What to look for
In practice mode, the replies are teaching fixtures, not real model results. In live mode, compare the direct and guided answers, check the JSON response, and use Python's two checks to verify the puzzles. The older model and either DeepSeek mode may get an answer right or wrong.

## Try it
1. In Step 1, keep the system prompt the same and write another architecture question. Which parts can be reused?
2. Remove the earlier answer from `follow_up` in Step 2. What context is missing?
3. Change the JSON decision in the practice reply to `API gateway`, then rerun Step 3. Which check fails?
4. In live mode, report the actual answers from both DeepSeek modes. Did the guided older-model prompt help? Repeat if you have time; these two examples are not a benchmark.

## Takeaway
A system message sets behavior, a user message asks for a task, and temporary context must be sent again. Validate outputs and test reasoning claims rather than assuming a model or prompt is always right.